# Paper 3 — Climate exposure under SSP3-7.0 multi-model ensemble

Nature Energy paper. Runs the calibrated household-energy ABM for Newcastle
across three Climate-DT realizations of SSP3-7.0 (ICON, IFS-FESOM, IFS-NEMO),
2020–2040, and reports the multi-model envelope of city-level demand.

**Climate inputs** live in the sibling `climate-to-abm` repo, schema:
`(latitude, longitude, timestamp, temp_C)`. Validated by `climate-to-abm validate`
before this notebook is run.

**Calibrated config** auto-detected from newest `results/calibration_*/`.

## Skeleton status

This is a runnable skeleton — the analysis cells (figures, exports) are scaffolded
with TODOs. The expensive parts (parallel ABM runs, ensemble aggregation) are wired.


## Imports

In [ ]:
from __future__ import annotations

from itertools import product
from pathlib import Path
import hashlib
import multiprocessing as mp
import random
import tempfile
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from household_energy.model import EnergyModel

warnings.filterwarnings('ignore', message='.*GeoSeries.notna.*')
warnings.filterwarnings('ignore', category=FutureWarning)


## 1) Settings

In [ ]:
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

# Newcastle synthpop + sociodemographic merge (same as paper1/paper2)
GEOJSON  = ROOT / 'data' / 'epc_abm_newcastle.geojson'
HIDP_CSV = ROOT / 'data' / 'hidp_uprn_matches_tiered.csv'

# Climate-DT projections live in the sibling preprocessor repo
CLIMATE_REPO = Path('/Users/abeltran/Documents/GitHub/climate-to-abm/data')
CLIMATE_MODELS = {
    'icon':      CLIMATE_REPO / 'newcastle_climate_dt_ssp370_icon_2t_2020_2040.parquet',
    'ifs_fesom': CLIMATE_REPO / 'newcastle_climate_dt_ssp370_ifs_fesom_2t_2020_2040.parquet',
    'ifs_nemo':  CLIMATE_REPO / 'newcastle_climate_dt_ssp370_ifs_nemo_2t_2020_2040.parquet',
}
for name, path in CLIMATE_MODELS.items():
    if not path.exists():
        raise FileNotFoundError(f'{name}: {path}')

OUTDIR = ROOT / 'notebooks' / 'results' / 'paper3_climate'
OUTDIR.mkdir(parents=True, exist_ok=True)

# ── Runtime ──────────────────────────────────────────────────────────────────
# Full projection window (2020-01-01 → 2040-01-01) is ~175,320 hours. Use
# fast-annualization for iteration; switch USE_FAST off for paper-quality figures.
USE_FAST = True
WINDOW_DAYS = 28
START_UTC   = '2025-01-01T00:00:00Z'   # mid-window starting point for fast mode
N_PROCS     = max(1, min(8, (mp.cpu_count() or 2) - 1))

WINDOW_HOURS  = (WINDOW_DAYS * 24) if USE_FAST else (20 * 365 * 24)
ANNUAL_FACTOR = (365.0 / WINDOW_DAYS) if USE_FAST else (1.0 / 20.0)

PROCESS_COLUMN = 'lsoa_code'

# ── Calibration auto-pickup ──────────────────────────────────────────────────
_cal_dirs = sorted((ROOT / 'results').glob('calibration_*'), key=lambda p: p.name)
if not _cal_dirs:
    raise FileNotFoundError(f"No calibration_* directory under {ROOT / 'results'}")
CAL_DIR = _cal_dirs[-1]
CAL_YAML = CAL_DIR / 'calibrated_config.yaml'
if not CAL_YAML.exists():
    raise FileNotFoundError(f"Calibrated config not found: {CAL_YAML}")
with CAL_YAML.open() as _f:
    CALIBRATED_CFG: dict = yaml.safe_load(_f) or {}
print(f'Calibration base: {CAL_DIR.name}')


def _deep_merge(base: dict, override: dict) -> dict:
    out = dict(base)
    for k, v in (override or {}).items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = _deep_merge(out[k], v)
        else:
            out[k] = v
    return out


# ── Aesthetics ────────────────────────────────────────────────────────────────
MODEL_COLORS = {'icon': '#4c78a8', 'ifs_fesom': '#54a24b', 'ifs_nemo': '#f58518'}
MODEL_LABELS = {'icon': 'ICON', 'ifs_fesom': 'IFS-FESOM', 'ifs_nemo': 'IFS-NEMO'}
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
                     'axes.spines.top': False, 'axes.spines.right': False})

print(f'Models:            {list(CLIMATE_MODELS.keys())}')
print(f'Workers:           {N_PROCS}')
print(f'Window:            {WINDOW_HOURS}h  (annualization ×{ANNUAL_FACTOR:.3f})')
print(f'Output:            {OUTDIR}')


## 2) Load and shard the synthetic population

Same enrichment as paper1/paper2 — EPC GeoJSON merged with HIDP sociodemographic
columns. LSOA-sharded parquets enable parallel execution across ~185 shards.

In [ ]:
def load_enriched_gdf(geojson_path: Path, hidp_csv_path: Path | None) -> gpd.GeoDataFrame:
    g = gpd.read_file(geojson_path)
    g['UPRN'] = g['UPRN'].astype(str).str.strip()
    if hidp_csv_path and hidp_csv_path.exists():
        hidp = pd.read_csv(hidp_csv_path, low_memory=False)
        hidp.columns = [c.strip() for c in hidp.columns]
        hidp['uprn_chr'] = hidp['uprn_chr'].astype(str).str.strip()
        hidp = hidp.drop_duplicates(subset=['uprn_chr'])
        g = g.merge(hidp, how='left', left_on='UPRN', right_on='uprn_chr',
                    suffixes=('_geo', '_hidp'))
        for base in ['lsoa_code', 'ward_code']:
            geo_col, hidp_col = f'{base}_geo', f'{base}_hidp'
            if base not in g.columns:
                if geo_col in g.columns and hidp_col in g.columns:
                    g[base] = g[geo_col].combine_first(g[hidp_col])
                elif geo_col in g.columns:
                    g[base] = g[geo_col]
                elif hidp_col in g.columns:
                    g[base] = g[hidp_col]
    return g


def to_wgs84_points(gdf_in: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    g = gdf_in.dropna(subset=['geometry']).copy()
    g = g.to_crs(4326) if g.crs else g.set_crs(4326)
    if (g.geometry.geom_type != 'Point').any():
        g['geometry'] = g.geometry.centroid
    return g


gdf_all = to_wgs84_points(load_enriched_gdf(GEOJSON, HIDP_CSV if HIDP_CSV.exists() else None))
gdf_all['AgentID'] = gdf_all['UPRN'].astype(str)

LSOA_UNITS = (
    gdf_all[PROCESS_COLUMN].astype(str).replace('', np.nan).dropna().unique().tolist()
)

UNIT_INPUT_DIR = OUTDIR / 'unit_inputs'
UNIT_INPUT_DIR.mkdir(exist_ok=True)
written = 0
for unit in LSOA_UNITS:
    fpath = UNIT_INPUT_DIR / f'{PROCESS_COLUMN}={unit}.parquet'
    if not fpath.exists():
        gdf_all[gdf_all[PROCESS_COLUMN].astype(str) == unit].to_parquet(fpath, index=False)
        written += 1

print(f'City:              {len(gdf_all):,} dwellings | {len(LSOA_UNITS)} LSOAs')
print(f'Parquets:          {written} newly written')


## 3) Model runner

`_run_lsoa_model` runs the calibrated ABM for one (LSOA × climate-model) cell,
returning annualized city-share demand. Seeded deterministically across kernel
restarts via sha256.

No policy intervention — Paper 3 is exposure of the **existing** stock to
climate uncertainty, so adoption is held at zero.

In [ ]:
def _build_and_run(gdf_in: gpd.GeoDataFrame, climate_parquet: Path,
                   cfg_dict: dict | None) -> float:
    merged = _deep_merge(CALIBRATED_CFG, cfg_dict or {})
    with tempfile.NamedTemporaryFile('w', suffix='.yaml', delete=False) as tmp:
        yaml.safe_dump(merged, tmp)
        cfg_path = tmp.name

    m = EnergyModel(
        gdf=gdf_in,
        climate_parquet=str(climate_parquet),
        climate_start=START_UTC,
        collect_agent_level=False,
        agent_collect_every=168,
        config_path=cfg_path,
    )
    for _ in range(WINDOW_HOURS):
        m.step()

    return float(sum(
        sum((getattr(h, 'annual_kwh_by_year', {}) or {}).values())
        for h in m.household_agents
    ))


def _run_lsoa_model(unit_val: str, model_name: str) -> dict:
    g = gpd.read_parquet(UNIT_INPUT_DIR / f'{PROCESS_COLUMN}={unit_val}.parquet')
    g['AgentID'] = g['UPRN'].astype(str)

    seed_key = f'paper3|{unit_val}|{model_name}'.encode()
    seed = int.from_bytes(hashlib.sha256(seed_key).digest()[:4], 'big')
    np.random.seed(seed); random.seed(seed)

    cfg = {'meta': {'name': f'paper3_{model_name}'},
           'model': {'heatpump_adoption_rate': 0.0}}
    kwh_window = _build_and_run(g, CLIMATE_MODELS[model_name], cfg)

    return {
        'unit': unit_val,
        'model': model_name,
        'kwh_window': kwh_window,
        'kwh_year_equiv': kwh_window * ANNUAL_FACTOR,
    }


def _run_lsoa_model_star(args):
    return _run_lsoa_model(*args)


def _run_parallel(jobs, n_procs: int):
    if n_procs <= 1 or not jobs:
        return [_run_lsoa_model_star(j) for j in jobs]
    try:
        ctx = mp.get_context('fork')
        with ctx.Pool(processes=n_procs) as pool:
            return pool.map(_run_lsoa_model_star, jobs)
    except Exception as e:
        print(f'Parallel pool failed ({e}); falling back to serial.')
        return [_run_lsoa_model_star(j) for j in jobs]


## 4) Run the three-model ensemble

One (LSOA × model) job per pool worker. Results cached to parquet; re-running
the cell skips any (LSOA, model) combination already present.

In [ ]:
ensemble_cache = OUTDIR / 'ensemble_raw.parquet'
all_jobs = [(unit, model) for unit in LSOA_UNITS for model in CLIMATE_MODELS.keys()]

if ensemble_cache.exists():
    cached = pd.read_parquet(ensemble_cache)
    done = set(zip(cached['unit'].astype(str), cached['model']))
    missing = [j for j in all_jobs if j not in done]
    if not missing:
        print(f'Cache complete — loading {ensemble_cache}')
        ensemble_raw = cached
    else:
        print(f'Cache partial — {len(missing):,} jobs to run...')
        new_rows = _run_parallel(missing, N_PROCS)
        ensemble_raw = pd.concat([cached, pd.DataFrame(new_rows)], ignore_index=True)
        ensemble_raw.to_parquet(ensemble_cache, index=False)
else:
    print(f'Running ensemble: {len(all_jobs):,} jobs across {N_PROCS} workers...')
    rows = _run_parallel(all_jobs, N_PROCS)
    ensemble_raw = pd.DataFrame(rows)
    ensemble_raw.to_parquet(ensemble_cache, index=False)

city = (
    ensemble_raw
    .groupby('model', as_index=False)
    .agg(city_kwh_year=('kwh_year_equiv', 'sum'),
         n_lsoas      =('unit',           'nunique'))
)
city['city_gwh_year'] = city['city_kwh_year'] / 1e6
city.to_csv(OUTDIR / 'ensemble_city_summary.csv', index=False)
print(city.to_string(index=False))


## 5) Figure — multi-model demand envelope (TODO)

Headline figure for Paper 3. Bar chart of city-level annual demand under each
of the three GCM realizations, with the multi-model mean line and ±1σ envelope.

When `USE_FAST=False` and the run covers 2020–2040 hour-by-hour, expand this to
a time-series plot: annual demand trajectories with the inter-model band shaded.

In [ ]:
# TODO: time-series figure once full 2020-2040 runs are available.
# Skeleton bar chart for fast mode:

fig, ax = plt.subplots(figsize=(7, 4.5))
order = list(CLIMATE_MODELS.keys())
vals = [float(city.loc[city['model'] == m, 'city_gwh_year'].iloc[0]) for m in order]
colors = [MODEL_COLORS[m] for m in order]

bars = ax.bar(range(len(order)), vals, color=colors, width=0.6, edgecolor='white')
mean = float(np.mean(vals))
std  = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
ax.axhline(mean, ls='--', color='#444', lw=1.0, label=f'Ensemble mean ({mean:.1f} GWh/yr)')
ax.axhspan(mean - std, mean + std, color='#999', alpha=0.15, label=f'±1σ ({std:.2f} GWh/yr)')

for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + max(vals)*0.01,
            f'{v:.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(range(len(order)))
ax.set_xticklabels([MODEL_LABELS[m] for m in order])
ax.set_ylabel('City annual demand (GWh/yr)')
ax.set_title(f'Newcastle demand under SSP3-7.0 — three-model ensemble'
             f'\n(window: {WINDOW_HOURS}h, start {START_UTC})', fontsize=10)
ax.legend(frameon=False, fontsize=9)
ax.set_ylim(0, max(vals) * 1.18)
fig.savefig(OUTDIR / 'fig1_ensemble_envelope.png')
plt.show()


## 6) Exposure distribution (TODO)

Per-dwelling distribution of climate-driven demand change under each model,
broken out by income quintile and EPC band. This is the equity dimension of the
exposure story — who is most exposed to the warmest/coolest realizations.

Inputs needed:
- Agent-level kwh output (set `collect_agent_level=True` in `_build_and_run` or
  add a second pass that captures per-agent annuals)
- Merge against `hh_income_band` and `sap_band_ord` from `gdf_all`


In [ ]:
# TODO: agent-level exposure distribution
# Requires re-running with collect_agent_level=True or post-hoc capture of
# household_agents' annual_kwh_by_year inside _build_and_run.


## 7) Export

In [ ]:
print('Outputs:')
for f in sorted(OUTDIR.glob('*.csv')):
    print(f'  {f.name}')
for f in sorted(OUTDIR.glob('*.png')):
    print(f'  {f.name}')
